# Upload SpliceLoc-ML to GitHub
Run each cell **one by one** in order.

**Before you start:**
1. You need a GitHub Personal Access Token (PAT)
2. Go to: https://github.com/settings/tokens
3. Click **Generate new token (classic)**
4. Give it a name like `SpliceLoc-ML upload`
5. Check the **repo** checkbox (full control of repositories)
6. Click **Generate token** and copy it — you will paste it below
7. **Keep it secret — do not share it with anyone**

## Step 1 — Enter your GitHub credentials

In [ ]:
# Fill in your details here
GITHUB_USERNAME = 'wedadmurar'
GITHUB_TOKEN    = 'paste_your_token_here'   # paste your token between the quotes
REPO_NAME       = 'SpliceLoc-ML'

print('Credentials set.')

## Step 2 — Install required library

In [ ]:
!pip install PyGithub -q
print('PyGithub installed.')

## Step 3 — Mount Google Drive (where your files are saved)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

## Step 4 — Set the path to your SpliceLoc-ML folder in Google Drive
Change the path below to wherever you saved your SpliceLoc-ML folder in Google Drive.

In [ ]:
import os

# Change this path to where your SpliceLoc-ML folder is in Google Drive
# Example: '/content/drive/MyDrive/SpliceLoc-ML'
LOCAL_FOLDER = '/content/drive/MyDrive/SpliceLoc-ML'

# Check the folder exists and list what is inside
if os.path.exists(LOCAL_FOLDER):
    print(f'Folder found: {LOCAL_FOLDER}')
    for root, dirs, files in os.walk(LOCAL_FOLDER):
        level = root.replace(LOCAL_FOLDER, '').count(os.sep)
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = '  ' * (level + 1)
        for file in files:
            filepath = os.path.join(root, file)
            size_kb = os.path.getsize(filepath) / 1024
            print(f'{subindent}{file}  ({size_kb:.1f} KB)')
else:
    print(f'ERROR: Folder not found at {LOCAL_FOLDER}')
    print('Please update LOCAL_FOLDER path above and run again.')

## Step 5 — Connect to GitHub and upload all files

In [ ]:
from github import Github
import base64

# Connect to GitHub
g = Github(GITHUB_TOKEN)
user = g.get_user()
repo = user.get_repo(REPO_NAME)
print(f'Connected to repository: {repo.full_name}')

# Get all existing files in the repo (to update instead of create if they already exist)
existing_files = {}
try:
    contents = repo.get_contents('')
    stack = list(contents)
    while stack:
        item = stack.pop(0)
        if item.type == 'dir':
            stack.extend(repo.get_contents(item.path))
        else:
            existing_files[item.path] = item.sha
    print(f'Found {len(existing_files)} existing files in repository.')
except Exception as e:
    print(f'Repository appears empty or error reading: {e}')

# Upload all files
uploaded = []
errors = []

for root, dirs, files in os.walk(LOCAL_FOLDER):
    # Skip hidden folders
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    
    for filename in files:
        # Skip hidden files and checkpoints
        if filename.startswith('.') or '.ipynb_checkpoints' in root:
            continue
            
        local_path = os.path.join(root, filename)
        # Create the GitHub path (relative to repo root)
        github_path = os.path.relpath(local_path, LOCAL_FOLDER).replace('\\', '/')
        
        try:
            # Read file as bytes
            with open(local_path, 'rb') as f:
                content = f.read()
            
            file_size_mb = len(content) / (1024 * 1024)
            
            # GitHub has a 100MB limit per file
            if file_size_mb > 95:
                print(f'SKIP (too large {file_size_mb:.1f}MB): {github_path}')
                continue
            
            if github_path in existing_files:
                # Update existing file
                repo.update_file(
                    path=github_path,
                    message=f'Update {github_path}',
                    content=content,
                    sha=existing_files[github_path]
                )
                print(f'Updated: {github_path}  ({file_size_mb:.2f} MB)')
            else:
                # Create new file
                repo.create_file(
                    path=github_path,
                    message=f'Add {github_path}',
                    content=content
                )
                print(f'Uploaded: {github_path}  ({file_size_mb:.2f} MB)')
            
            uploaded.append(github_path)
            
        except Exception as e:
            print(f'ERROR on {github_path}: {e}')
            errors.append((github_path, str(e)))

print(f'\n── Upload Complete ──')
print(f'Successfully uploaded: {len(uploaded)} files')
if errors:
    print(f'Errors: {len(errors)}')
    for path, err in errors:
        print(f'  {path}: {err}')
print(f'\nView your repository at: https://github.com/{GITHUB_USERNAME}/{REPO_NAME}')

## Step 6 — Verify everything uploaded correctly

In [ ]:
# List all files now in the repository
print(f'Files currently in {REPO_NAME}:\n')
contents = repo.get_contents('')
stack = list(contents)
all_files = []
while stack:
    item = stack.pop(0)
    if item.type == 'dir':
        stack.extend(repo.get_contents(item.path))
    else:
        all_files.append(item.path)
        print(f'  {item.path}')

print(f'\nTotal: {len(all_files)} files')
print(f'\nRepository URL: https://github.com/{GITHUB_USERNAME}/{REPO_NAME}')